### Importing Required Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

### Loading Data

In [3]:
df = pd.read_csv("data\dataset_with_all_features.csv")
df["date"] = pd.to_datetime(df["date"])

In [4]:
df.head()

,county,date,fire_label,max_frp,max_brightness,fire_count,temp_max,temp_min,humidity,wind_speed,...,temp_max_7d_rolling_mean,humidity_7d_rolling_mean,temp_max_14d_rolling_mean,humidity_14d_rolling_mean,temp_max_30d_rolling_mean,humidity_30d_rolling_mean,temperature_anomaly,vpd,wind_speed_drought_interaction,temp_max_humidity_interaction
0,Alameda,2010-01-01,0.0,0.0,0.0,0.0,13.2815,5.7315,84.409645,6.792466,...,13.281500,84.409645,13.281500,84.409645,13.281500,84.409645,0.000000,0.237843,0.000000,207.063300
1,Alameda,2010-01-02,0.0,0.0,0.0,0.0,13.4815,6.1815,95.559820,6.479999,...,13.381500,89.984733,13.381500,89.984733,13.381500,89.984733,0.100000,0.068628,0.000000,59.860287
2,Alameda,2010-01-03,0.0,0.0,0.0,0.0,12.9815,2.8315,84.136185,10.739832,...,13.248167,88.035217,13.248167,88.035217,13.248167,88.035217,-0.266667,0.237316,10.739832,205.936114
3,Alameda,2010-01-04,0.0,0.0,0.0,0.0,13.6315,2.5315,79.060646,7.244860,...,13.344000,85.791574,13.344000,85.791574,13.344000,85.791574,0.287500,0.326816,14.489720,285.434804
4,Alameda,2010-01-05,0.0,0.0,0.0,0.0,13.7315,4.6815,69.012860,7.100310,...,13.421500,82.435831,13.421500,82.435831,13.421500,82.435831,0.310000,0.486796,21.300930,425.499913


### Sort (IMPORTANT for time-based ops)

In [5]:
df = df.sort_values(["county", "date"]).reset_index(drop=True)

### MISSING VALUES (KNN Imputer)

In [6]:
impute_cols = [
    "temp_max",
    "temp_min",
    "humidity",
    "wind_speed",
    "precipitation",
    "drought_index",
    "temp_max_7d_rolling_mean",
    "humidity_7d_rolling_mean",
    "temp_max_14d_rolling_mean",
    "humidity_14d_rolling_mean",
    "temp_max_30d_rolling_mean",
    "humidity_30d_rolling_mean",
    "temperature_anomaly",
    "vpd",
    "wind_speed_drought_interaction",
    "temp_max_humidity_interaction"
]

imputer = KNNImputer(n_neighbors=5)
df[impute_cols] = imputer.fit_transform(df[impute_cols])

### OUTLIER HANDLING (IQR CAPPING)

In [7]:
def iqr_cap(dataframe, col):
    q1 = dataframe[col].quantile(0.25)
    q3 = dataframe[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    dataframe[col] = np.clip(dataframe[col], lower, upper)
    return dataframe

for col in ["temp_max", "wind_speed"]:
    df = iqr_cap(df, col)

### TIME-BASED SPLIT (STRICT)

In [8]:
train = df[df["date"] < "2018-01-01"].copy()
val = df[(df["date"] >= "2018-01-01") & (df["date"] < "2020-01-01")].copy()
test = df[df["date"] >= "2020-01-01"].copy()

print("Train shape:", train.shape)
print("Val shape:", val.shape)
print("Test shape:", test.shape)

Train shape: (169476, 28)
Val shape: (42340, 28)
Test shape: (21228, 28)


### ENCODING (One-hot county)

In [9]:
train = pd.get_dummies(train, columns=["county"], drop_first=True)
val = pd.get_dummies(val, columns=["county"], drop_first=True)
test = pd.get_dummies(test, columns=["county"], drop_first=True)

# Align columns so val/test match train
train, val = train.align(val, join="left", axis=1, fill_value=0)
train, test = train.align(test, join="left", axis=1, fill_value=0)

In [16]:
train.shape, val.shape, test.shape

((169476, 84), (42340, 84), (21228, 84))

### SPLIT FEATURES / TARGET

In [10]:
target = "fire_label"

drop_cols = [
    "date",
    "fire_label",
    "max_frp",
    "max_brightness",
    "fire_count"
]

X_train = train.drop(columns=drop_cols, errors="ignore")
y_train = train[target]

X_val = val.drop(columns=drop_cols, errors="ignore")
y_val = val[target]

X_test = test.drop(columns=drop_cols, errors="ignore")
y_test = test[target]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (169476, 79)
X_val shape: (42340, 79)
X_test shape: (21228, 79)


### SCALING (ONLY TRAIN FIT)

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")

# save scaled versions as DataFrames
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_val_scaled_df = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

### SMOTE (TRAIN ONLY)

In [12]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Original y_train counts:")
print(y_train.value_counts())

print("\nSMOTE y_train counts:")
print(pd.Series(y_train_sm).value_counts())

Original y_train counts:
fire_label
0.0    167787
1.0      1689
Name: count, dtype: int64

SMOTE y_train counts:
fire_label
0.0    167787
1.0    167787
Name: count, dtype: int64


In [13]:
X_train_scaled_sm, y_train_scaled_sm = smote.fit_resample(X_train_scaled_df, y_train)

### SAVE OUTPUTS

In [14]:
# Original time-based splits
train.to_csv("train.csv", index=False)
val.to_csv("val.csv", index=False)
test.to_csv("test.csv", index=False)

In [15]:
pd.concat(
    [
        pd.DataFrame(X_train_sm, columns=X_train.columns),
        pd.Series(y_train_sm, name="fire_label")
    ],
    axis=1
).to_csv("train_features_smote.csv", index=False)
